In [2]:
# run first time
import sys
!{sys.executable} -m pip install "gymnasium[box2d]"==1.2.3 stable_baselines3 tensorflow matplotlib numpy
# !{sys.executable} -m pip install "gymnasium[box2d]" imageio statsmodels pyvirtualdisplay tensorflow matplotlib numpy "imageio[ffmpeg]"


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: /Users/wangrunyuan/.pyenv/versions/3.11.9/bin/python -m pip install --upgrade pip


In [58]:
import time
from collections import deque, namedtuple

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import utils
import matplotlib.pyplot as plt
from bidirection_lunar_lander import BidirectionalLunarLander

from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
import gymnasium as gym

In [2]:
def navie_inverted_controller(obs, phase, sign):
    x = obs[0]
    y = obs[1]
    vx = obs[2]
    vy = obs[3]
    theta = obs[4]
    omega = obs[5]

    # angle wrap to [-pi, pi]
    theta_wrapped = np.arctan2(np.sin(theta), np.cos(theta))
    theta_abs = abs(theta_wrapped)

    side = 0
    main = 0
    if phase == 1:
        if y > 1.4:
            return np.array([0, 0], dtype=np.float32), phase
            
        if theta_abs < 1.9:
            side = -0.6 * sign
            main = 0.8
        elif theta_abs > 1.9 and abs(omega) > 0.1 :
            side = 1 * sign
            main = -1
        else:
            phase = 2

    if phase == 2:
        if y > 1.4:
            return np.array([0, 0], dtype=np.float32), phase

        main = -0.6
        if vx > 0.1:
            side = 0.5
        elif vx < -0.1:
            side = -0.5
        else:
            side = 0

    return np.array([
        np.clip(main, -1.0, 1.0),
        np.clip(side, -1.0, 1.0)
    ], dtype=np.float32), phase

def getInitSign(vx0):
    if vx0 > 0:
        return 1
    else:
        return -1

In [3]:
class InvertedController:
    def __init__(self):
        self.phase = 0
        self.sign = 1

    def reset(self, vx0):
        self.phase = 1
        self.sign = getInitSign(vx0)

    def act(self, obs):
        action, phase = navie_inverted_controller(obs, self.phase, self.sign)
        self.phase = phase
        return action, phase

In [10]:
##### See what happens by calling expert act in 500 steps
env = BidirectionalLunarLander(continuous=True, render_mode="human")
obs, info = env.reset()

controller = InvertedController()
controller.reset(obs[2])

for step in range(500):
    action, phase = controller.act(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    x = obs[0]
    vx = obs[2]
    theta = obs[4]
    omega = obs[5]
    theta_w = np.arctan2(np.sin(theta), np.cos(theta), )
    # print(f"\rphase: {phase} action: {action} x: {x}, vx: {vx} theta: {theta_w}, omega: {omega}")

    if terminated or truncated:
        print(f"Episode ended: {step}")
        break

env.close

Episode ended: 124


<bound method LunarLander.close of <bidirection_lunar_lander.BidirectionalLunarLander object at 0x149e6d890>>

In [4]:
class InvertedHoverWrapper(gym.Wrapper):
    def __init__(self, env, controller, y_ref=1.2):
        super().__init__(env)
        self.controller = controller
        self.y_ref = y_ref

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.controller.reset(obs[2])

        for _ in range(200):
            action, phase = self.controller.act(obs)
            obs, _, terminated, truncated, _ = self.env.step(action)

            if terminated or truncated:
                obs, info = self.env.reset(**kwargs)
                self.controller.reset(obs[2])

            if phase == 2:
                break

        return obs, info

    def step(self, action):
        obs, _, terminated, truncated, info = self.env.step(action)

        reward = self.tracking_reward(obs, action)

        done = terminated or truncated
        if abs(obs[1]) > 3.0:
            done = True

        return obs, reward, done, False, info

    def tracking_reward(self, obs, action):
        x, y, vx, vy, theta, omega, _, _ = obs
        main, side = action
    
        theta_err = np.arctan2(
            np.sin(theta - np.pi),
            np.cos(theta - np.pi)
        )
    
        reward = (
            - 4.0 * theta_err**2
            - 2.0 * omega**2
    
            - 3.0 * (y - self.y_ref)**2
            - 2.0 * vy**2
    
            - 3.0 * x**2
            - 2.0 * vx**2
    
            - 0.1 * main**2
            - 0.05 * side**2
        )
    
        return reward

In [7]:
def make_env(render=None):
    base_env = BidirectionalLunarLander(
        continuous=True,
        render_mode=render
    )
    controller = InvertedController()
    wrapped = InvertedHoverWrapper(base_env, controller)
    return wrapped

env = DummyVecEnv([make_env])

model = SAC(
    policy="MlpPolicy",
    env=env,
    learning_rate=3e-4,
    buffer_size=200_000,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    train_freq=1,
    gradient_steps=1,
    learning_starts=5_000,
    verbose=0
)

model.learn(total_timesteps=1000_000)
model.save("sac_inverted_hover")

In [36]:
class HybridExpert:
    def __init__(self, inverted_controller, rf_policy):
        self.inverted_controller = inverted_controller
        self.rf_policy = rf_policy
        self.phase = 0

    def reset(self, vx0):
        self.inverted_controller.reset(vx0)
        self.phase = 0

    def act(self, obs):
        if self.phase != 2:
            action, phase = self.inverted_controller.act(obs)
            self.phase = phase
            return action
        else:
            action, _ = self.rf_policy.predict(obs, deterministic=True)
            return action


In [47]:
##### Verify HybridExpert's capability in 500 steps
env = BidirectionalLunarLander(continuous=True, render_mode="human")
controller = InvertedController()
rf_policy = SAC.load("sac_inverted_hover")
hyperExpert = HybridExpert(controller, rf_policy)

obs, _ = env.reset()
hyperExpert.reset(obs[2])

for i in range(500):
    action = hyperExpert.act(obs)
    obs, _, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
        break

In [ ]:
########################################################

In [44]:
def collect_one_episode(env, expert, max_steps=500):
    obs, info = env.reset()
    expert.reset(obs[2])
    
    episode = []
    for t in range(max_steps):
        action = expert.act(obs)

        episode.append({
            "obs": obs.copy(),
            "action": action.copy()
        })

        obs, reward, terminated, truncated, info = env.step(action)

        if terminated or truncated:
            break

    return episode


In [45]:
def collect_expert_episodes(
    env,
    expert,
    num_episodes=300,
    min_length=30,
    require_inverted=True
):
    expert_episodes = []

    for ep in range(num_episodes):
        episode = collect_one_episode(env, expert)

        if len(episode) < min_length:
            continue

        expert_episodes.append([
            {
                "obs": step["obs"],
                "action": step["action"]
            }
            for step in episode
        ])

        print(
            f"Accepted episode {len(expert_episodes)} "
            f"(len={len(episode)})"
        )

    print(f"\nCollected {len(expert_episodes)} expert episodes")
    return expert_episodes


In [46]:
def save_expert_episodes(expert_episodes, path="expert_lander.npz"):
    obs = []
    actions = []
    episode_lens = []

    for ep in expert_episodes:
        episode_lens.append(len(ep))
        for step in ep:
            obs.append(step["obs"])
            actions.append(step["action"])

    obs = np.array(obs, dtype=np.float32)
    actions = np.array(actions, dtype=np.float32)
    episode_lens = np.array(episode_lens, dtype=np.int32)

    np.savez(
        path,
        obs=obs,
        actions=actions,
        episode_lens=episode_lens
    )

    print(
        f"Saved {len(episode_lens)} episodes, "
        f"{len(obs)} total steps"
    )


In [47]:
env = BidirectionalLunarLander(continuous=True)
controller = InvertedController()
rf_policy = SAC.load("sac_inverted_hover")
hyperExpert = HybridExpert(controller, rf_policy)

expert_episodes = collect_expert_episodes(
    env,
    hyperExpert,
    num_episodes=1000,
    min_length=200
)

save_expert_episodes(expert_episodes, path="expert_lander.npz")

Accepted episode 1 (len=228)
Accepted episode 2 (len=500)
Accepted episode 3 (len=225)
Accepted episode 4 (len=414)
Accepted episode 5 (len=500)
Accepted episode 6 (len=247)
Accepted episode 7 (len=254)
Accepted episode 8 (len=500)
Accepted episode 9 (len=234)
Accepted episode 10 (len=260)
Accepted episode 11 (len=500)
Accepted episode 12 (len=241)
Accepted episode 13 (len=209)
Accepted episode 14 (len=500)
Accepted episode 15 (len=232)
Accepted episode 16 (len=216)
Accepted episode 17 (len=500)
Accepted episode 18 (len=500)
Accepted episode 19 (len=256)
Accepted episode 20 (len=236)
Accepted episode 21 (len=500)
Accepted episode 22 (len=298)
Accepted episode 23 (len=206)
Accepted episode 24 (len=286)
Accepted episode 25 (len=500)
Accepted episode 26 (len=500)
Accepted episode 27 (len=378)
Accepted episode 28 (len=500)
Accepted episode 29 (len=425)
Accepted episode 30 (len=215)
Accepted episode 31 (len=228)
Accepted episode 32 (len=214)
Accepted episode 33 (len=241)
Accepted episode 34

In [55]:
# Sanity check
data = np.load("expert_lander.npz")

obs_all = data["obs"]
actions_all = data["actions"]
episode_lens = data["episode_lens"]

env = BidirectionalLunarLander(
    continuous=True,
    render_mode="human"
)

ep_id = np.random.randint(len(episode_lens))

start = sum(episode_lens[:ep_id])
end = start + episode_lens[ep_id]

obs, _ = env.reset()

for t in range(start, end):
    obs, _, terminated, truncated, _ = env.step(actions_all[t])
    if terminated or truncated:
        break


In [59]:
def build_bc_policy(obs_dim=8, act_dim=2):
    inputs = layers.Input(shape=(obs_dim,))
    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(act_dim, activation="tanh")(x)

    model = models.Model(inputs, outputs)
    return model

In [60]:
data = np.load("expert_lander.npz")

obs_all = data["obs"]          # (N, 8)
actions_all = data["actions"]  # (N, 2)

dataset = tf.data.Dataset.from_tensor_slices(
    (obs_all.astype(np.float32), actions_all.astype(np.float32))
)

dataset = dataset.shuffle(10000).batch(256).prefetch(tf.data.AUTOTUNE)

policy = build_bc_policy()

policy.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss="mse"
)

policy.fit(
    dataset,
    epochs=30
)


Epoch 1/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 1s 484us/step - loss: 0.1253
Epoch 2/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 478us/step - loss: 0.0863
Epoch 3/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 468us/step - loss: 0.0783
Epoch 4/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 458us/step - loss: 0.0734
Epoch 5/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 474us/step - loss: 0.0699
Epoch 6/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 459us/step - loss: 0.0666
Epoch 7/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 468us/step - loss: 0.0639
Epoch 8/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 484us/step - loss: 0.0616
Epoch 9/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 478us/step - loss: 0.0594
Epoch 10/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 491us/step - loss: 0.0577
Epoch 11/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 487us/step - loss: 0.0558
Epoch 12/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 501us/step - loss: 0.0543
Epoch 13/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 488us/step - loss: 0.0531
Epoch 14/30
716/716 ━━━━━━━━━━━━━━━━━━━━ 0s 487us/step - loss: 0.0519
Epoch 15/30
716/716 ━━━━━━━━━

In [66]:
env = BidirectionalLunarLander(
    continuous=True,
    render_mode="human"
)

obs, _ = env.reset()

for t in range(500):
    obs_batch = obs.reshape(1, -1).astype(np.float32)
    action = policy(obs_batch, training=False).numpy()[0]

    obs, _, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
        break

In [73]:
def dagger_rollout(
    env,
    policy,
    expert,
    max_steps=500
):
    obs, _ = env.reset()
    expert.reset(obs[2])

    rollout_data = []

    for t in range(max_steps):
        # learner action
        obs_batch = obs.reshape(1, -1).astype(np.float32)
        learner_action = policy(obs_batch, training=False).numpy()[0]

        # expert label
        expert_action = expert.act(obs)

        rollout_data.append({
            "obs": obs.copy(),
            "action": expert_action.copy()
        })

        obs, _, terminated, truncated, _ = env.step(learner_action)

        if terminated or truncated:
            break

    return rollout_data


In [74]:
def aggregate_dataset(dataset, new_data):
    for step in new_data:
        dataset.append((
            step["obs"].astype(np.float32),
            step["action"].astype(np.float32)
        ))


In [75]:
dataset = list(zip(obs_all, actions_all))

In [76]:
def train_policy_tf(policy, dataset, epochs=5):
    obs = np.array([d[0] for d in dataset], dtype=np.float32)
    actions = np.array([d[1] for d in dataset], dtype=np.float32)

    ds = tf.data.Dataset.from_tensor_slices((obs, actions))
    ds = ds.shuffle(10000).batch(256).prefetch(tf.data.AUTOTUNE)

    policy.fit(ds, epochs=epochs, verbose=0)


In [77]:
env = BidirectionalLunarLander(continuous=True, render_mode="human")
controller = InvertedController()
rf_policy = SAC.load("sac_inverted_hover")
hyperExpert = HybridExpert(controller, rf_policy)

# dataset 已由 BC 初始化
for iteration in range(10):
    print(f"\nDAgger iteration {iteration}")

    rollout_data = dagger_rollout(
        env,
        policy,
        hyperExpert,
        max_steps=300
    )

    aggregate_dataset(dataset, rollout_data)

    train_policy_tf(
        policy,
        dataset,
        epochs=5
    )



DAgger iteration 0

DAgger iteration 1

DAgger iteration 2

DAgger iteration 3

DAgger iteration 4

DAgger iteration 5

DAgger iteration 6

DAgger iteration 7

DAgger iteration 8

DAgger iteration 9


In [ ]:
env = BidirectionalLunarLander(
    continuous=True,
    render_mode="human"
)

obs, _ = env.reset()

for t in range(500):
    obs_batch = obs.reshape(1, -1).astype(np.float32)
    action = policy(obs_batch, training=False).numpy()[0]

    obs, _, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
        break

In [ ]:
##########################